# Bronze Layer: NEFT/RTGS Timestamp Fixer

This script fixes the missing `txn_ts` column in the `neft_rtgs_transactions` bronze table by inferring a random timestamp between the historically observed `card_transactions` and `upi_transactions`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType
from datetime import datetime

CATALOG = "bfsi"
SCHEMA = "bronze_layer"
CARD_TABLE = f"{CATALOG}.{SCHEMA}.card_transactions"
UPI_TABLE = f"{CATALOG}.{SCHEMA}.upi_transactions"
NEFT_TABLE = f"{CATALOG}.{SCHEMA}.neft_rtgs_transactions"

In [0]:
print(f"Extracting min and max txn_ts from {CARD_TABLE} and {UPI_TABLE}")

df_card = spark.read.table(CARD_TABLE)
df_upi = spark.read.table(UPI_TABLE)

card_bounds = df_card.agg(F.min("txn_ts").cast("long").alias("min_ts"), F.max("txn_ts").cast("long").alias("max_ts")).collect()[0]
upi_bounds = df_upi.agg(F.min("txn_ts").cast("long").alias("min_ts"), F.max("txn_ts").cast("long").alias("max_ts")).collect()[0]

min_ts_card, max_ts_card = card_bounds['min_ts'], card_bounds['max_ts']
min_ts_upi, max_ts_upi = upi_bounds['min_ts'], upi_bounds['max_ts']

# Fallback just in case empty tables
min_ts_card = min_ts_card if min_ts_card is not None else int(datetime(2023, 1, 1).timestamp())
max_ts_card = max_ts_card if max_ts_card is not None else int(datetime(2023, 12, 31).timestamp())
min_ts_upi = min_ts_upi if min_ts_upi is not None else int(datetime(2023, 1, 1).timestamp())
max_ts_upi = max_ts_upi if max_ts_upi is not None else int(datetime(2023, 12, 31).timestamp())

global_min_ts = min(min_ts_card, min_ts_upi)
global_max_ts = max(max_ts_card, max_ts_upi)

print(f"Global Min Timestamp: {datetime.fromtimestamp(global_min_ts)}")
print(f"Global Max Timestamp: {datetime.fromtimestamp(global_max_ts)}")

In [0]:
print(f"Adding txn_ts to {NEFT_TABLE}")

df_neft = spark.read.table(NEFT_TABLE)

# Prevent adding the column multiple times by replacing if it exists
if "txn_ts" in df_neft.columns:
    df_neft = df_neft.drop("txn_ts")

# Create random timestamp expression
rand_expr = F.expr(f"timestamp({global_min_ts} + cast(rand() * ({global_max_ts} - {global_min_ts}) as bigint))")

df_neft_fixed = df_neft.withColumn("txn_ts", rand_expr)

df_neft_fixed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(NEFT_TABLE)

print(f"Successfully added txn_ts to NEFT/RTGS transactions.")
display(spark.read.table(NEFT_TABLE).select("amount", "txn_ts").limit(5))